# Ripple tank interference

*waves · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/waves/ripple_tank)

Two dippers bob in step on a water surface and send out circular ripples. The 2D wave equation is stepped with the Laplacian as a $3\times3$ convolution kernel (conv2d), a damping border swallows the outgoing waves, and the interference pattern settles into still nodal lines where the path difference is a half-odd number of wavelengths.

**Vector form**

$$\frac{\partial^2 u}{\partial t^2} + \gamma(\mathbf r)\,\frac{\partial u}{\partial t} = c^2\,\nabla^2 u + f(\mathbf r, t),\qquad |r_1 - r_2| = (m + \tfrac12)\,\lambda$$

**Index (tensor) form**

$$(\nabla^2 u)_{ab} = \frac{1}{h^2}\,K_{\alpha\beta}\,u_{a+\alpha,\,b+\beta},\qquad K = \begin{pmatrix}0&1&0\\1&-4&1\\0&1&0\end{pmatrix},\qquad \frac{c\,\Delta t}{h} \le \frac{1}{\sqrt2}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $u_{ab}$ | `u` | (n, n) | surface height at grid cell $(a, b)$ |
| $K_{\alpha\beta}$ | `stencil(dev)` | (1, 1, 3, 3) | five-point Laplacian kernel, offsets $\alpha, \beta \in \{-1, 0, 1\}$ |
| $K_{\alpha\beta}u_{a+\alpha,b+\beta}$ | `conv2d(u, K, padding=1)` | (n, n) | repeated $\alpha, \beta$ ⇒ summed: the kernel slides over the grid |
| $\gamma_{ab}$ | `damping(n, border, dev)` | (n, n) | zero inside, rising smoothly in the absorbing border |
| $f_{ab}(t)$ | `A * sin(omega * t) * blobs` | (n, n) | two small Gaussian sources oscillating in phase |
| $\Delta t$ | `time_step(h, c, courant)` | scalar | Courant number above $1/\sqrt2$ is refused (the scheme would blow up) |
| $\langle u^2\rangle$ | `intensity` | (n, n) | running average of $u^2$ over about one period: dark lines are nodes |

Units: cell size $h = 1$, wave speed $c = 1$. Leap-frog in time with the damping term centred, so the step is explicit. Along a nodal line the ripple from one source always arrives half a period after the one from the other, so they cancel; the dashed curves are the hyperbolas $|r_1 - r_2| = (m + \tfrac12)\lambda$, which match the dark lines in the right panel far enough from the sources. Bringing the sources further apart than $\lambda$ adds more nodal lines. The border is a crude absorber (a friction layer), not a perfectly matched layer, so a little reflection survives. Right panel shows the running mean of $u^2$; the first few periods are transient while the waves fill the tank.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/waves/ripple_tank/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli, device

from . import equation, plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    lam, c, h = params["wavelength"], tensors.C, tensors.H
    X, Y = tensors.grid(dev)
    K = tensors.stencil(dev)
    gamma = tensors.damping(dev)
    blob, s1, s2 = tensors.sources(X, Y, params["separation"])
    dt = equation.time_step(h, c, params["courant"])

    times, u, intensity = solve.solve(blob, gamma, K, lam, c, h, dt, params["t_max"])

    D = equation.path_difference(X, Y, s1, s2)
    print(f"time step dt = {dt:.3f}  (Courant number {c * dt / h:.2f} <= 1/sqrt(2))")
    print(f"sources {params['separation']:g} cells apart = {params['separation'] / lam:.2f} wavelengths")
    far = Y > tensors.SOURCE_Y + 3 * lam  # away from the near field of the sources
    I_node, I_anti = solve.nodal_contrast(intensity[-1], D, lam, (gamma == 0) & far)
    print(f"mean intensity on predicted nodal lines {I_node:.3g}, on antinodal lines {I_anti:.3g}")
    return plot.animate(times, u, intensity, D, s1, s2, lam, tensors.BORDER, out_dir / "ripple_tank.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/waves/ripple_tank/equation.py
r"""Physics → tensor translation.

The damped, driven 2D wave equation

    \partial_t^2 u + \gamma \partial_t u = c^2 \nabla^2 u + f

with the five-point Laplacian written as a 3 × 3 kernel K sliding over the grid (α, β summed):

    (\nabla^2 u)_{ab} = \frac{1}{h^2} K_{\alpha\beta} u_{a+\alpha, b+\beta}        ← conv2d(u, K, padding=1)

Leap-frog in time with the damping centred, (u^{n+1} − u^{n−1}) / 2Δt:

    (1 + \gamma\Delta t/2) u^{n+1} = 2u^n - (1 - \gamma\Delta t/2) u^{n-1} + \Delta t^2 (c^2 \nabla^2 u^n + f^n)

The explicit scheme is stable only for c Δt / h ≤ 1/√2 (Courant–Friedrichs–Lewy in 2D).
"""

import math

import torch
import torch.nn.functional as F


def laplacian(u: torch.Tensor, K: torch.Tensor, h: float) -> torch.Tensor:
    r"""(∇²u)_{ab} = K_{αβ} u_{a+α, b+β} / h².   u: (n, n), K: (1, 1, 3, 3) → (n, n); zero outside the grid."""
    return F.conv2d(u[None, None], K, padding=1)[0, 0] / h**2


def time_step(h: float, c: float, courant: float) -> float:
    """Δt = S h / c, refusing S > 1/√2 where the explicit 2D scheme blows up."""
    if not 0 < courant <= 1 / math.sqrt(2):
        raise ValueError(f"Courant number {courant} violates the 2D CFL limit 1/sqrt(2) = {1 / math.sqrt(2):.4f}")
    return courant * h / c


def step(u, u_prev, f, gamma, K, c: float, h: float, dt: float) -> torch.Tensor:
    """One damped leap-frog step. All fields (n, n)."""
    a = 0.5 * gamma * dt
    return (2 * u - (1 - a) * u_prev + dt**2 * (c**2 * laplacian(u, K, h) + f)) / (1 + a)


def path_difference(X: torch.Tensor, Y: torch.Tensor, s1: tuple, s2: tuple) -> torch.Tensor:
    """|r1 − r2|: difference of the distances to the two sources at every cell.   (n, n)"""
    r1 = torch.sqrt((X - s1[0]) ** 2 + (Y - s1[1]) ** 2)
    r2 = torch.sqrt((X - s2[0]) ** 2 + (Y - s2[1]) ** 2)
    return (r1 - r2).abs()


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/waves/ripple_tank/tensors.py
"""Build the concrete tensors: the Laplacian kernel, the damping border and the two sources."""

import torch

N = 200  # cells per side
BORDER = 30  # absorbing border width (cells)
SOURCE_Y = 60.0  # height of the two sources
H = 1.0  # cell size
C = 1.0  # wave speed


def grid(device: torch.device) -> tuple[torch.Tensor, torch.Tensor]:
    """Cell centres X, Y of shape (n, n), indexed [a, b] = [x, y]."""
    s = torch.arange(N, device=device, dtype=torch.float32) * H
    return torch.meshgrid(s, s, indexing="ij")


def stencil(device: torch.device) -> torch.Tensor:
    """K_{αβ} of shape (1, 1, 3, 3): the five-point Laplacian (out/in channel dimensions for conv2d)."""
    K = torch.tensor([[0.0, 1.0, 0.0], [1.0, -4.0, 1.0], [0.0, 1.0, 0.0]], device=device)
    return K.view(1, 1, 3, 3)


def damping(device: torch.device, strength: float = 0.25) -> torch.Tensor:
    """γ_{ab} (n, n): 0 inside, growing like (depth into the border)² up to `strength` at the walls."""
    s = torch.arange(N, device=device, dtype=torch.float32)
    depth = torch.clamp(BORDER - torch.minimum(s, N - 1 - s), min=0) / BORDER  # (n,) 0 inside → 1 at the wall
    d2 = torch.maximum(depth[:, None], depth[None, :])
    return strength * d2**2


def sources(X: torch.Tensor, Y: torch.Tensor, separation: float, width: float = 1.5):
    """Two Gaussian blobs (n, n) summed into one tensor, and their centres."""
    x_mid = (N - 1) * H / 2
    s1, s2 = (x_mid - separation / 2, SOURCE_Y), (x_mid + separation / 2, SOURCE_Y)
    blob = sum(torch.exp(-((X - sx) ** 2 + (Y - sy) ** 2) / (2 * width**2)) for sx, sy in (s1, s2))
    return blob, s1, s2


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/waves/ripple_tank/solve.py
"""March the ripple tank forward and keep a running average of u² (the intensity)."""

import math

import torch

from problems._common.progress import steps

from .equation import step


def solve(blob, gamma, K, wavelength: float, c: float, h: float, dt: float, t_max: float, n_frames: int = 120):
    """Return (times (F,), u (F, n, n), intensity (F, n, n)).

    The sources add f = A sin(ωt) · blob with ω = 2πc/λ. The intensity is an exponential running mean of u²
    with a memory of about one period, so it settles to ⟨u²⟩ once the tank is full.
    """
    omega = 2 * math.pi * c / wavelength
    period = wavelength / c
    n_steps = int(round(t_max / dt))
    save_every = max(1, n_steps // (n_frames - 1))
    u = torch.zeros_like(blob)
    u_prev = torch.zeros_like(blob)
    intensity = torch.zeros_like(blob)
    times, us, Is = [0.0], [u], [intensity]
    for n in steps(n_steps, "time steps"):
        f = omega**2 * math.sin(omega * n * dt) * blob  # amplitude ∝ ω² keeps the ripple height similar for any λ
        u, u_prev = step(u, u_prev, f, gamma, K, c, h, dt), u
        intensity = intensity + (u**2 - intensity) * (dt / period)
        if (n + 1) % save_every == 0:
            times.append((n + 1) * dt)
            us.append(u)
            Is.append(intensity)
    return torch.tensor(times), torch.stack(us), torch.stack(Is)


def nodal_contrast(intensity: torch.Tensor, path_diff: torch.Tensor, wavelength: float, region: torch.Tensor):
    """Mean ⟨u²⟩ where |r1 − r2| ≈ (m + ½)λ (predicted nodes) and where |r1 − r2| ≈ mλ (antinodes), inside region."""
    z = path_diff / wavelength
    nodes = region & ((z - 0.5) - torch.round(z - 0.5)).abs().lt(0.08)
    antinodes = region & (z - torch.round(z)).abs().lt(0.08)
    return intensity[nodes].mean().item(), intensity[antinodes].mean().item()


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/waves/ripple_tank/plot.py
"""Left: the ripples. Right: the running intensity ⟨u²⟩, where the nodal lines appear dark. Dashed: predicted nodes."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from problems._common.plotting import frame_indices, save_gif


def animate(times, u, intensity, path_diff, s1, s2, wavelength: float, border: int, path: Path) -> Path:
    t = times.cpu().numpy()
    inner = slice(border, u.shape[-1] - border)  # hide the absorbing border
    U = u[:, inner, inner].cpu().numpy()  # (F, m, m)
    W = intensity[:, inner, inner].cpu().numpy()  # running ⟨u²⟩
    D = path_diff[inner, inner].cpu().numpy()
    lo, hi = border, u.shape[-1] - border
    extent = (lo, hi, lo, hi)
    u_lim = np.quantile(np.abs(U[-1]), 0.98) + 1e-9
    I_lim = np.quantile(W[-1], 0.98) + 1e-9
    m_max = int(D.max() / wavelength - 0.5)
    levels = [(m + 0.5) * wavelength for m in range(m_max + 1)]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7, 3.7))
    im1 = ax1.imshow(
        U[0].T,
        origin="lower",
        extent=extent,
        cmap=plt.get_cmap("RdBu_r", 10),
        vmin=-u_lim,
        vmax=u_lim,
        interpolation="nearest",
    )
    im2 = ax2.imshow(
        W[0].T,
        origin="lower",
        extent=extent,
        cmap=plt.get_cmap("Blues", 8),
        vmin=0,
        vmax=I_lim,
        interpolation="nearest",
    )
    for ax, name in ((ax1, "surface u"), (ax2, "intensity ⟨u²⟩")):
        if levels:
            ax.contour(
                np.arange(lo, hi) + 0.5, np.arange(lo, hi) + 0.5, D.T, levels=levels, colors="#f59e0b",
                linestyles="--", linewidths=0.9,
            )  # fmt: skip
        ax.plot([s1[0], s2[0]], [s1[1], s2[1]], "o", color="#111827", ms=4)
        ax.set_title(name, fontsize=10)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.grid(False)
    title = fig.suptitle("two coherent sources   t = 0", fontsize=11)

    def update(step: int):
        im1.set_data(U[step].T)
        im2.set_data(W[step].T)
        title.set_text(f"two coherent sources   λ = {wavelength:g}   t = {t[step]:.0f}")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(t), 60), path, fps=10)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'wavelength': 16.0,
    'separation': 48.0,
    'courant': 0.5,
    't_max': 300.0
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.waves.ripple_tank")]:
    del sys.modules[name]
main = importlib.import_module("problems.waves.ripple_tank.main")
problem_dir = Path("problems/waves/ripple_tank")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
